In [1]:
# Resolve paths consistently when launched from the root or notebooks/.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "topic_embeddings").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the derStandard project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Evaluate topic analogies

For **A : B :: C : D**, rank topics by cosine similarity to **B − A + C** in the normalized embedding space. A, B, and C are excluded; D is only the expected answer.

Run all cells to evaluate every row in `data/benchmarks/topic_analogy_candidates.csv` and see the final top-5, top-3, and top-1 counts. The same evaluator powers the standalone check and batch results. Ties preserve embedding-file order. Validation failures are reported and count as non-hits in the overall summary.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from topic_embeddings.evaluate_analogies import AnalogyEvaluator, summarize_analogies

DATA_PATH = Path("results/sgns/topic_embeddings_normalized.csv")
ANALOGY_PATH = Path("data/benchmarks/topic_analogy_candidates.csv")
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]

# Load topic embeddings
topics = pd.read_csv(DATA_PATH, keep_default_na=False, dtype={"Topic_Id": str})
embedding_columns = sorted(c for c in topics.columns if c.startswith("embedding_"))
print(f"Embedding file: {DATA_PATH.resolve()}")
print(f"Loaded {len(topics):,} topics × {len(embedding_columns)} dimensions.")

# Load analogies
analogies = pd.read_csv(ANALOGY_PATH, keep_default_na=False, dtype=str)
print(f"Loaded {len(analogies):,} analogies from {ANALOGY_PATH}.")

# Initialize the analogy evaluator
evaluator = AnalogyEvaluator(topics, analogies)


Embedding file: /Users/nachodoradollamas/Downloads/derStandard/results/sgns/topic_embeddings_normalized.csv
Loaded 1,000 topics × 150 dimensions.
Loaded 173 analogies from data/benchmarks/topic_analogy_candidates.csv.


## Manually check one analogy

In [3]:
TOP_K = 5
for i in range(173, 174):
    ANALOGY_ID = f"AN{i:03d}"
    single_metrics, single_neighbors = evaluator.evaluate_analogy(ANALOGY_ID, top_k=TOP_K, verbose=True)


AN173 | Relation: hierarchy_shortening | Label: structural
Notes: (none)
A (query input): ID=135 | wissenschaft/natur/klima
B (query input): ID=330 | wissenschaft/klima
C (query input): ID=487 | sport/mehrsport/schach
D (expected answer): ID=582 | sport/schach


 Rank Topic_Id                          Topic_path  Cosine_similarity  Expected_answer
    1      582                        sport/schach           0.642920             True
    2      756                      etat/sosindwir           0.528502            False
    3      454                      etat/tv/serien           0.467112            False
    4      515 panorama/oesterreich/laenderchronik           0.454087            False
    5      144               sport/fussball/wm2022           0.451697            False


## Evaluate every analogy

`all_results` contains expected-answer ranks, hit flags, and errors for each analogy. For example, inspect misses with `all_results.loc[~all_results["top5"]]`. The batch is independent of the standalone ID and neighbor count above.

In [4]:
all_results = evaluator.evaluate_all_analogies(analogies)
errors = all_results.loc[all_results["error"].ne(""), ["analogy_id", "error"]]
print(f"Evaluated {len(all_results)} analogies: {len(all_results) - len(errors)} successful, {len(errors)} errors.")
if not errors.empty:
    print("\nValidation errors (included as non-hits):")
    print(errors.to_string(index=False))

summary = summarize_analogies(all_results)
print("\nExpected answers found in the top results:")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

Evaluated 173 analogies: 173 successful, 0 errors.

Expected answers found in the top results:
Cutoff  Matches  Total analogies  Hit rate (%)
 Top 5      125              173         72.25
 Top 3      110              173         63.58
 Top 1       79              173         45.66
